# **Level 1 (Basic) — Task 1: Data Cleaning & Preprocessing**

**Internship:** Codveda Virtual Data Science Internship  
**Level:** 1 — Basic | **Task:** 1 of 2  
**Tools:** Python, Pandas, NumPy, Scikit-learn  

---

## Project Overview

This notebook covers the full data cleaning and preprocessing pipeline for the telecom customer churn dataset.
The raw data comes in two separate files representing an 80/20 train-test split, which we merge here for unified cleaning.

**Dataset:** `churn-bigml-80.csv` (2,666 rows) + `churn-bigml-20.csv` (667 rows) → merged: **3,333 rows × 20 columns**

**Task Objectives:**
- Handle missing values
- Detect and investigate potential outliers
- Encode categorical variables (International plan, Voice mail plan, Churn)
- Engineer new features to enrich the dataset for downstream EDA and modelling
- Normalize numerical columns

**Notebook Structure:**
1. Environment Setup
2. Phase 1 — Loading & Merging
3. Phase 2 — Data Sanitization
4. Phase 3 — Feature Engineering & Encoding
5. Phase 4 — Normalization *(pending)*
6. Final Checks & Summary


## 1. Environment Setup

We begin by connecting to Google Drive (where the datasets are stored) and importing the core libraries.

- **`pandas`** — data loading, manipulation, and inspection
- **`numpy`** — numerical operations and conditional column creation (`np.where`, `np.select`)


In [ ]:
#Connect google drive to colab notebook
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## 2. Phase 1 — Loading & Merging the Dataset

Before any cleaning, we need to understand what we're working with.
We load each file separately and inspect them individually — checking shape, column types, and a few sample rows — before combining them.
This lets us catch any structural differences between the two files early.

**Why label `data_type` before merging?**
Once two DataFrames are stacked into one, there's no way to tell which row came from which file.
Adding a `'train'`/`'test'` label *before* concatenation preserves that provenance.
This matters if you ever want to reconstruct the original split for modelling.


In [ ]:
#read in the two churn datasets
churn_1 = pd.read_csv('/content/drive/MyDrive/Codveda task 1(Level1(Basic)-Data Cleaning & Preprocessing)/churn-bigml-80.csv')
churn_2 = pd.read_csv('/content/drive/MyDrive/Codveda task 1(Level1(Basic)-Data Cleaning & Preprocessing)/churn-bigml-20.csv')

In [ ]:
#preview and overview of churn 1
print(churn_1.info())
churn_1.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [ ]:
#preview and overview of churn 2
print(churn_2.info())
churn_2.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 667 entries, 0 to 666
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   667 non-null    object 
 1   Account length          667 non-null    int64  
 2   Area code               667 non-null    int64  
 3   International plan      667 non-null    object 
 4   Voice mail plan         667 non-null    object 
 5   Number vmail messages   667 non-null    int64  
 6   Total day minutes       667 non-null    float64
 7   Total day calls         667 non-null    int64  
 8   Total day charge        667 non-null    float64
 9   Total eve minutes       667 non-null    float64
 10  Total eve calls         667 non-null    int64  
 11  Total eve charge        667 non-null    float64
 12  Total night minutes     667 non-null    float64
 13  Total night calls       667 non-null    int64  
 14  Total night charge      667 non-null    fl

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,LA,117,408,No,No,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,False
1,IN,65,415,No,No,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,True
2,NY,161,415,No,No,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,True
3,SC,111,415,No,No,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,False
4,HI,49,510,No,No,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,False


In [ ]:
#Add a column data_type to both ('train' and 'test') before merging so you can easily split them later without confusion.
churn_1['data_type'] = 'train'
churn_2['data_type'] = 'test'

In [ ]:
#merge churn datasets to clean and engineer features
df = pd.concat([churn_1, churn_2], axis=0).reset_index(drop=True)

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   object 
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   object 
 4   Voice mail plan         3333 non-null   object 
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,data_type
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,...,16.78,244.7,91,11.01,10.0,3,2.70,1,False,train
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,...,16.62,254.4,103,11.45,13.7,3,3.70,1,False,train
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,...,10.30,162.6,104,7.32,12.2,5,3.29,0,False,train
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,...,5.26,196.9,89,8.86,6.6,7,1.78,2,False,train
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,...,12.61,186.9,121,8.41,10.1,3,2.73,3,False,train


**Merge result:** 3,333 rows × 21 columns — 20 original features + the `data_type` tracking column. The index has been reset so rows are numbered 0–3332 continuously. No structural differences were found between the two files — same columns, same dtypes throughout.


## 3. Phase 2 — Data Sanitization

This phase ensures the data is trustworthy before any analytical work begins. We check three things:

1. **Missing values** — any NaN/null entries that would break calculations or distort statistics
2. **Data type correctness** — columns stored in the wrong type (e.g. phone codes stored as integers)
3. **Logical consistency** — rows that are internally inconsistent (e.g. charges recorded against zero minutes)


In [ ]:
#Handle missing values
df.isnull().sum()

,0
State,0
Account length,0
Area code,0
International plan,0
Voice mail plan,0
Number vmail messages,0
Total day minutes,0
Total day calls,0
Total day charge,0
Total eve minutes,0


> **Result:** All 21 columns return 0 missing values across all 3,333 rows. This is a clean dataset out of the box — no imputation needed.


In [ ]:
#convert 'Area code' from integer to string
df['Area code'] = df['Area code'].astype(str)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   object 
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   object 
 3   International plan      3333 non-null   object 
 4   Voice mail plan         3333 non-null   object 
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   

**Why convert `Area code` to string?**
Area codes are identifiers, not quantities — 408, 415, and 510 are labels, not measurements.
Leaving them as integers risks two problems:
(1) a normalization step would scale them, which is meaningless for a categorical code, and
(2) a correlation matrix would treat them as numeric, producing spurious correlations.
Casting to `str` signals to every downstream process: *this column is a label, not a number*.


In [ ]:
# Ensure "Yes/No" or State codes are consistent (no weird spaces or casing issues like "yes" vs "Yes").
df['Voice mail plan'].unique()

array(['Yes', 'No'], dtype=object)

> **Result:** `Voice mail plan` contains only `'Yes'` and `'No'` — no casing inconsistencies, no unexpected values.

### Outlier Investigation — Logical Consistency Checks

Rather than purely statistical outlier detection, we apply domain knowledge here.
In a telecom billing dataset, a *charge with zero minutes* is a billing anomaly — it means a customer was charged for calls they didn't make.
We check all four billing categories (day, evening, night, international) for this condition.


In [ ]:
#Identify rows where Minutes are 0 but Charges are $>0$ (or vice versa) to decide if they are outliers.
df_day = df[(df['Total day minutes'] == 0) & (df['Total day charge'] > 0)]
df_day.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,data_type


In [ ]:
df_eve = df[(df['Total eve minutes'] == 0) & (df['Total eve charge'] > 0)]
df_eve.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,data_type


In [ ]:
df_night = df[(df['Total night minutes'] == 0) & (df['Total night charge'] > 0)]
df_night.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,data_type


In [ ]:
df_night = df[(df['Total intl minutes'] == 0) & (df['Total intl charge'] > 0)]
df_night.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,data_type


> **Result:** All four checks return empty DataFrames. No rows have charges without corresponding minutes.
The dataset is logically consistent — no billing anomalies detected.

> **Note:** A standard duplicate rows check (`df.duplicated().sum()`) is also recommended in a complete sanitization pipeline. Worth adding for future reference.


## 4. Phase 3 — Feature Engineering & Encoding

The original 20 columns tell us *what happened* (minutes used, charges incurred) but not much about *what kind of customer this is*.
Feature engineering transforms and combines existing columns into new ones that carry richer analytical signal.

**New features created:**

| Feature | Type | What it captures |
|---|---|---|
| `day_rate`, `eve_rate`, `night_rate`, `intl_rate` | Numeric | Price per minute by billing period |
| `is_loyal` | Binary | Whether account length exceeds the loyalty threshold |
| `vmail_user_type` | Categorical | How the customer uses (or doesn't) their voicemail plan |
| `service_sentiment` | Categorical | Customer friction level based on service call frequency |
| `total_overall_charge` | Numeric | Total bill across all billing periods |
| `total_minutes` | Numeric | Total call volume across all periods |
| `value_score` | Numeric | Effective rate — total spend ÷ total minutes |

We also binary-encode `International plan`, `Voice mail plan`, and `Churn` as integers (0/1) for compatibility with correlation matrices and ML pipelines.


In [ ]:
#The "Rate" Engineering (Price Sensitivity)
df['day_rate'] = df['Total day charge'] / df['Total day minutes']
df['eve_rate'] = df['Total eve charge'] / df['Total eve minutes']
df['night_rate'] = df['Total night charge'] / df['Total night minutes']
df['intl_rate'] = df['Total intl charge'] / df['Total intl minutes']

# List of the rate columns we created
rate_cols = ['day_rate', 'eve_rate', 'night_rate', 'intl_rate']

# Filling all NaN values (from 0/0 divisions) with 0
df[rate_cols] = df[rate_cols].fillna(0)

**Why create rate columns?**
Absolute totals (minutes used, charges incurred) don't tell the full story.
Rate = Charge ÷ Minutes gives the *price per minute* — a normalised cost efficiency measure.
Two customers can have the same total charge but very different per-minute rates depending on usage volume.

**The NaN fix:** When both minutes and charges are 0 (no activity in that period), the division produces NaN (0 ÷ 0 is undefined). Filling with 0 is semantically correct — a rate of 0 means no usage and no charge.


In [ ]:
#The "Loyalty" Threshold we use 110 days as a loyalty cutoff. In the industry, we call this "Tenure Binning."
df['is_loyal'] = np.where(df['Account length'] > 110, 1, 0)

In [ ]:
df['is_loyal'].value_counts()

,count
is_loyal,
0,1984
1,1349


> **Output:** 1,984 customers are `0` (account ≤ 110 days) vs. 1,349 classified as `1` (account > 110 days) — roughly a 60/40 split.





In [ ]:
#The "Ghost User" Logic: Create vmail_user_type: Categorize into Active_User, Wasteful_Payer, or Standard based on your vmail_plan and vmail_messages logic.
conditions = [
    (df['Voice mail plan'] == 'Yes') & (df['Number vmail messages'] == 0),  # Wasteful
    (df['Voice mail plan'] == 'Yes') & (df['Number vmail messages'] > 0),   # Active
    (df['Voice mail plan'] == 'No') & (df['Number vmail messages'] == 0),   # Standard
    (df['Voice mail plan'] == 'No') & (df['Number vmail messages'] > 0)     # Anomaly
]

# Define the labels for those conditions
values = ['Wasteful_Payer', 'Active_User', 'Standard', 'Anomaly']

# Create the new column
df['vmail_user_type'] = np.select(conditions, values, default='Unknown')

In [ ]:
df['vmail_user_type'].unique()

array(['Active_User', 'Standard'], dtype=object)

> **Output:** Only `'Active_User'` and `'Standard'` appear in the data.
This means: every customer *with* a voicemail plan also uses it (no `Wasteful_Payer`), and no customer *without* a plan has voicemail messages (no `Anomaly`).
This is actually a useful data insight — voicemail uptake is clean and internally consistent in this dataset.


In [ ]:
#The "service sentiment":Create service_sentiment: Map customer_service_calls to Normal (0-2), Nervous (3), and High_Risk (4+).
conditions = [
    (df['Customer service calls'] <= 2),
    (df['Customer service calls'] == 3),
    (df['Customer service calls'] >= 4)
    ]

# Define the labels for those conditions
values = ['Normal', 'Nervous', 'High_Risk']

# Create the new column
df['service_sentiment'] = np.select(conditions, values, default='Unknown')

In [ ]:
df['service_sentiment'].value_counts()

,count
service_sentiment,
Normal,2637
Nervous,429
High_Risk,267


> **Output:** Normal: 2,637 (79%) | Nervous: 429 (13%) | High_Risk: 267 (8%)

> The 267 High_Risk customers will be particularly interesting to examine in the EDA — high service call frequency is one of the stronger known indicators of churn in telecom datasets.


In [ ]:
# Create total_overall_charge: Sum of Day + Eve + Night + Intl charges. This helps see the "Bill Shock" factor.
df['total_overall_charge'] = df['Total day charge'] + df['Total eve charge'] + df['Total night charge'] + df['Total intl charge']

# Total Minutes: Summing all usage categories (Day, Eve, Night, Intl) to capture the customer's total "Volume of Activity" across all time buckets.
df['total_minutes'] = (df['Total day minutes'] + df['Total eve minutes'] + df['Total night minutes'] + df['Total intl minutes'])

# Value Score: Calculating the "Effective Rate" (Total Spend / Total Minutes).
# This measures cost-efficiency; customers with a high score are paying more per minute and may be more sensitive to competitor pricing.
df['value_score'] = df['total_overall_charge'] / df['total_minutes'].replace(0, np.nan)
df['value_score'] = df['value_score'].fillna(0)



**Why aggregate?**
`total_overall_charge` collapses all billing periods into a single 'total bill' — the number a customer actually sees on their statement. High total bills can drive churn.

`value_score` (effective rate = total charge ÷ total minutes) normalises spend by volume. A high value_score means the customer pays more per minute — a proxy for price sensitivity.

Using `.replace(0, np.nan)` before division avoids NaN from customers with zero total minutes, then `fillna(0)` handles those edge cases cleanly.


In [ ]:
# Mapping binary categories to integers
# We use .map() to ensure total control over which label becomes 1 (the positive case)
binary_mapping = {'yes': 1, 'no': 1}
churn_mapping = {True: 1, False: 0}

df['International plan'] = df['International plan'].str.lower().map({'yes': 1, 'no': 0})
df['Voice mail plan'] = df['Voice mail plan'].str.lower().map({'yes': 1, 'no': 0})
df['Churn'] = df['Churn'].map({True: 1, False: 0})

In [ ]:
# Convert mapped columns to integers so they show up in the correlation matrix
df['International plan'] = df['International plan'].astype(int)
df['Voice mail plan'] = df['Voice mail plan'].astype(int)
df['Churn'] = df['Churn'].astype(int)

**Why encode as integers?**
`.map()` with an explicit dictionary is the most transparent binary encoding method — you control exactly which label becomes 1.
The `.astype(int)` step is necessary because `.map()` can return `float64` in some cases, and integer dtype is required for clean correlation matrices and ML model inputs.




In [ ]:
df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,day_rate,eve_rate,night_rate,intl_rate,is_loyal,vmail_user_type,service_sentiment,total_overall_charge,total_minutes,value_score
0,KS,128,415,0,1,25,265.1,110,45.07,197.4,...,0.170011,0.085005,0.044994,0.270000,1,Active_User,Normal,75.56,717.2,0.105354
1,OH,107,415,0,1,26,161.6,123,27.47,195.5,...,0.169988,0.085013,0.045008,0.270073,0,Active_User,Normal,59.24,625.2,0.094754
2,NJ,137,415,0,0,0,243.4,114,41.38,121.2,...,0.170008,0.084983,0.045018,0.269672,1,Standard,Normal,62.29,539.4,0.115480
3,OH,84,408,1,0,0,299.4,71,50.90,61.9,...,0.170007,0.084976,0.044997,0.269697,0,Standard,Normal,66.80,564.8,0.118272
4,OK,75,415,1,0,0,166.7,113,28.34,148.3,...,0.170006,0.085030,0.044997,0.270297,0,Standard,Nervous,52.09,512.0,0.101738


## 5. Phase 4 — Normalization ⚠️ *(Add code here)*

> **This section is required by the task brief but is not yet implemented.**

The task objective includes: *normalize numerical columns*.
Normalization brings all numerical features onto a common scale. This matters because:
- Distance-based algorithms (KNN, SVM, clustering) are scale-sensitive — a column ranging 0–350 will dominate one ranging 0–1
- Gradient-descent models (Logistic Regression, Neural Networks) converge faster on scaled inputs
- Comparison of feature importance across columns with different ranges becomes valid

**Recommended approach:** `MinMaxScaler` (scales each column to [0, 1]) from `sklearn.preprocessing`.

In [ ]:
# Select numerical columns to normalise (exclude binary 0/1 columns and categoricals)
cols_to_scale = [
    'Account length', 'Number vmail messages',
    'Total day minutes', 'Total day calls', 'Total day charge',
    'Total eve minutes', 'Total eve calls', 'Total eve charge',
    'Total night minutes', 'Total night calls', 'Total night charge',
    'Total intl minutes', 'Total intl calls', 'Total intl charge',
    'Customer service calls', 'day_rate', 'eve_rate', 'night_rate',
    'intl_rate', 'total_overall_charge', 'total_minutes', 'value_score'
]

scaler = MinMaxScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print('Normalization complete.')
df[cols_to_scale].describe().round(3)

## 6. Final Checks & Summary

The cells below verify the final state of the dataset after all transformations have been applied.


In [ ]:
#final checks
print(f'The Shape of the cleaned dataset is {df.shape}')
print()
print(f'Missing values are: ')
print(df.isnull().sum())
print()
df.info()

The Shape of the cleaned dataset is (3333, 31)

Missing values are: 
State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
data_type                 0
day_rate                  0
eve_rate                  0
night_rate                0
intl_rate                 0
is_loyal                  0
vmail_user_type           0
service_sentiment         0
total_overall_charge      0
total_minutes             0
value_score               0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex:

## Summary

This notebook completed the following steps on the merged churn dataset (3,333 rows):

| Step | Action | Outcome |
|---|---|---|
| Loading | Merged two CSVs (80/20 split) | 3,333 rows × 21 columns |
| Missing Values | Checked all columns | 0 missing values found |
| Type Correction | Area code → string | Prevents erroneous numeric treatment |
| Consistency Check | Minutes=0 & Charge>0 across 4 billing types | No anomalies found |
| Feature Engineering | 11 new columns created | Dataset expanded to 31 columns |
| Encoding | International plan, Voice mail plan, Churn → binary int | ML-ready target and plan columns |
| Normalization | *(pending)* | To be added in Phase 4 |

**Output dataset:** `df` — 3,333 rows × 31 columns, ready for EDA (Level 1, Task 2).

---
*Codveda Data Science Internship — Level 1, Task 1 | Deborah Olofin*
